# Free-static target-entropy coarse-only diagnostics

This notebook audits the completed `free_static_target_entropy_coarse_only` run and compares it directly with the controlled full-token reference `free_static_target_entropy`. The graph, regularisation, temporal encoder, spatial module, structured-parallel predictor, optimiser, seed, data, and checkpoint-selection rule are held fixed; the controlled change is removal of the future `s2` objective and use of the frozen coarse decoder.

The notebook focuses on whether coarse-only training improves `s1` cross-entropy, top-k accuracy, probability assigned to the true token, generated code diversity, decoded forecasting performance, and graph learning. Run the cells in order. Edit paths and diagnostic sample sizes only in **Cell 2**.


In [ ]:
# Cell 1 — all imports and repository setup.

from __future__ import annotations

from collections import defaultdict
from pathlib import Path
from typing import Any, Mapping, Sequence

import copy
import json
import math
import os
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from IPython.display import display
from torch import Tensor, nn

# Locate the repository whether Jupyter was launched from the repository
# root or from notebooks/.
_working_directory = Path.cwd().resolve()
_repository_candidates = [
    _working_directory,
    _working_directory.parent,
    _working_directory.parent.parent,
]

REPO_ROOT = next(
    (
        candidate
        for candidate in _repository_candidates
        if (candidate / "src").is_dir()
        and (candidate / "configs").is_dir()
    ),
    None,
)

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the dissertation repository root. "
        "Launch Jupyter from the repository root or notebooks/."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.chdir(REPO_ROOT)

from src.data.cached_token_graph_dataset import CachedTokenGraphDataset
from src.data.load_candle_data import (
    clean_candle_splits,
    get_channel_index,
    load_candle_splits,
)
from src.data.token_graph_dataset import (
    load_origin_aligned_token_cache,
    validate_origin_aligned_token_cache,
)
from src.models.dynamic_graph.model import DynamicGraphTokenForecaster
from src.models.dynamic_graph.modules import aggregate_graph_values
from src.models.kronos_tokenizer import KronosTokenizerAdapter
from src.utils.config import load_yaml


In [ ]:
# Cell 2 — paths and diagnostic controls.

RUN_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/final_model/initial_test/"
    "free_static_target_entropy_coarse_only"
).expanduser().resolve()

REFERENCE_RUN_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/final_model/initial_test/"
    "free_static_target_entropy"
).expanduser().resolve()

TRAIN_CACHE_PATH = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/final_model/tokens/"
    "origin_aligned_train_tokens.pt"
).expanduser().resolve()

VALIDATION_CACHE_PATH = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/final_model/tokens/"
    "origin_aligned_val_tokens.pt"
).expanduser().resolve()

DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
).expanduser().resolve()

FORECASTING_CONFIG_PATH = REPO_ROOT / "configs" / "forecasting.yaml"

BEST_CHECKPOINT_PATH = RUN_DIR / "best_checkpoint.pt"
LAST_CHECKPOINT_PATH = RUN_DIR / "last_checkpoint.pt"
BEST_PREDICTIONS_PATH = RUN_DIR / "best_validation_predictions.pt"
BEST_GRAPHS_PATH = RUN_DIR / "best_validation_graphs.pt"
HISTORY_PATH = RUN_DIR / "history.csv"
METRIC_TABLE_PATH = RUN_DIR / "best_validation_metric_table.csv"
TOKEN_METRIC_PATH = RUN_DIR / "best_validation_token_metrics.csv"
DIAGNOSTICS_PATH = RUN_DIR / "best_validation_diagnostics.json"
RESOLVED_CONFIG_PATH = RUN_DIR / "resolved_config.json"
RUN_METADATA_PATH = RUN_DIR / "run_metadata.json"

REFERENCE_BEST_CHECKPOINT_PATH = (
    REFERENCE_RUN_DIR / "best_checkpoint.pt"
)
REFERENCE_BEST_PREDICTIONS_PATH = (
    REFERENCE_RUN_DIR / "best_validation_predictions.pt"
)
REFERENCE_BEST_GRAPHS_PATH = (
    REFERENCE_RUN_DIR / "best_validation_graphs.pt"
)
REFERENCE_HISTORY_PATH = REFERENCE_RUN_DIR / "history.csv"
REFERENCE_METRIC_TABLE_PATH = (
    REFERENCE_RUN_DIR / "best_validation_metric_table.csv"
)
REFERENCE_TOKEN_METRIC_PATH = (
    REFERENCE_RUN_DIR / "best_validation_token_metrics.csv"
)
REFERENCE_DIAGNOSTICS_PATH = (
    REFERENCE_RUN_DIR / "best_validation_diagnostics.json"
)
REFERENCE_RESOLVED_CONFIG_PATH = (
    REFERENCE_RUN_DIR / "resolved_config.json"
)

DIAGNOSTIC_SEED = 42
GRADIENT_BATCH_SIZE = 2

# Use all 380 September validation windows for definitive s1 token
# probability and vocabulary diagnostics. Set this back to 64 only if
# local runtime becomes prohibitive.
TOKEN_DIAGNOSTIC_WINDOWS = 380
TOKEN_DIAGNOSTIC_BATCH_SIZE = 2

DECODE_DIAGNOSTIC_WINDOWS = 6
DECODE_DIAGNOSTIC_ASSETS = 16
TOKENIZER_SERIES_BATCH_SIZE = 64

if torch.cuda.is_available():
    MODEL_DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    MODEL_DEVICE = torch.device("mps")
else:
    MODEL_DEVICE = torch.device("cpu")

required_paths = [
    RUN_DIR,
    REFERENCE_RUN_DIR,
    TRAIN_CACHE_PATH,
    VALIDATION_CACHE_PATH,
    DATA_DIR,
    FORECASTING_CONFIG_PATH,
    BEST_CHECKPOINT_PATH,
    LAST_CHECKPOINT_PATH,
    BEST_PREDICTIONS_PATH,
    BEST_GRAPHS_PATH,
    HISTORY_PATH,
    METRIC_TABLE_PATH,
    TOKEN_METRIC_PATH,
    DIAGNOSTICS_PATH,
    RESOLVED_CONFIG_PATH,
    RUN_METADATA_PATH,
    REFERENCE_BEST_CHECKPOINT_PATH,
    REFERENCE_BEST_PREDICTIONS_PATH,
    REFERENCE_BEST_GRAPHS_PATH,
    REFERENCE_HISTORY_PATH,
    REFERENCE_METRIC_TABLE_PATH,
    REFERENCE_TOKEN_METRIC_PATH,
    REFERENCE_DIAGNOSTICS_PATH,
    REFERENCE_RESOLVED_CONFIG_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

print("Repository:", REPO_ROOT)
print("Coarse-only run:", RUN_DIR)
print("Full-token reference:", REFERENCE_RUN_DIR)
print("Diagnostic device:", MODEL_DEVICE)
print("Token diagnostic windows:", TOKEN_DIAGNOSTIC_WINDOWS)
print(
    "Decoder diagnostic windows/assets:",
    (DECODE_DIAGNOSTIC_WINDOWS, DECODE_DIAGNOSTIC_ASSETS),
)


In [ ]:
# Cell 3 — load current/reference artefacts, caches, raw data, and models.

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)
last_checkpoint = torch.load(
    LAST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)
prediction_artifact = torch.load(
    BEST_PREDICTIONS_PATH,
    map_location="cpu",
    weights_only=False,
)
graph_artifact = torch.load(
    BEST_GRAPHS_PATH,
    map_location="cpu",
    weights_only=False,
)

reference_best_checkpoint = torch.load(
    REFERENCE_BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)
reference_prediction_artifact = torch.load(
    REFERENCE_BEST_PREDICTIONS_PATH,
    map_location="cpu",
    weights_only=False,
)
reference_graph_artifact = torch.load(
    REFERENCE_BEST_GRAPHS_PATH,
    map_location="cpu",
    weights_only=False,
)

history = pd.read_csv(HISTORY_PATH)
metric_table = pd.read_csv(METRIC_TABLE_PATH)
saved_token_metric_table = pd.read_csv(TOKEN_METRIC_PATH)

reference_history = pd.read_csv(REFERENCE_HISTORY_PATH)
reference_metric_table = pd.read_csv(REFERENCE_METRIC_TABLE_PATH)
reference_token_metric_table = pd.read_csv(REFERENCE_TOKEN_METRIC_PATH)

with DIAGNOSTICS_PATH.open("r", encoding="utf-8") as file:
    saved_diagnostics = json.load(file)

with RESOLVED_CONFIG_PATH.open("r", encoding="utf-8") as file:
    resolved_config_file = json.load(file)

with RUN_METADATA_PATH.open("r", encoding="utf-8") as file:
    run_metadata = json.load(file)

with REFERENCE_DIAGNOSTICS_PATH.open("r", encoding="utf-8") as file:
    reference_diagnostics = json.load(file)

with REFERENCE_RESOLVED_CONFIG_PATH.open("r", encoding="utf-8") as file:
    reference_resolved_config_file = json.load(file)

resolved_config = best_checkpoint["resolved_config"]
reference_resolved_config = reference_best_checkpoint["resolved_config"]

if resolved_config != resolved_config_file:
    raise AssertionError(
        "best_checkpoint.pt and resolved_config.json disagree."
    )

if last_checkpoint["resolved_config"] != resolved_config:
    raise AssertionError(
        "Best and last checkpoints use different resolved configurations."
    )

if reference_resolved_config != reference_resolved_config_file:
    raise AssertionError(
        "Reference best_checkpoint.pt and resolved_config.json disagree."
    )

model_config = resolved_config["models"]["dynamic_graph"]
reference_model_config = reference_resolved_config["models"]["dynamic_graph"]

if model_config["graph"]["type"] != "free_static":
    raise AssertionError(
        "This notebook requires graph.type='free_static'."
    )

if reference_model_config["graph"]["type"] != "free_static":
    raise AssertionError(
        "The reference run is not a free-static graph run."
    )

expected_regularisation = {
    "graph_entropy_reg": 0.0,
    "graph_target_entropy": 2.2,
    "graph_target_entropy_reg": 0.05,
    "graph_temporal_smooth_reg": 0.0,
}

for label, configuration in (
    ("coarse-only", model_config),
    ("full-token reference", reference_model_config),
):
    regularisation_config = configuration.get(
        "graph_regularisation",
        {},
    )

    for key, expected_value in expected_regularisation.items():
        observed_value = regularisation_config.get(key)

        if observed_value is None:
            raise AssertionError(
                f"{label} run is missing graph setting {key!r}."
            )

        if not math.isclose(
            float(observed_value),
            expected_value,
            rel_tol=0.0,
            abs_tol=1.0e-12,
        ):
            raise AssertionError(
                f"Unexpected {label} {key}. "
                f"Expected {expected_value}, observed {observed_value}."
            )

current_heads = model_config["heads"]
reference_heads = reference_model_config["heads"]

if current_heads["future_token_mode"] != "coarse_only":
    raise AssertionError(
        "Current run must use future_token_mode='coarse_only'."
    )

if not math.isclose(
    float(current_heads["s2_loss_weight"]),
    0.0,
    rel_tol=0.0,
    abs_tol=1.0e-12,
):
    raise AssertionError(
        "Current run must use s2_loss_weight=0."
    )

if reference_heads["future_token_mode"] != "full":
    raise AssertionError(
        "Reference run must use future_token_mode='full'."
    )

if not math.isclose(
    float(reference_heads["s2_loss_weight"]),
    1.0,
    rel_tol=0.0,
    abs_tol=1.0e-12,
):
    raise AssertionError(
        "Reference run must use s2_loss_weight=1."
    )

for key in (
    "context_length",
    "d_model",
    "num_st_blocks",
    "temporal",
    "graph",
    "future_predictor",
    "loss",
):
    if model_config[key] != reference_model_config[key]:
        raise AssertionError(
            f"Controlled comparison changed model setting {key!r}."
        )

prediction_result = prediction_artifact["prediction_result"]
graph_artifacts = graph_artifact["graph_artifacts"]

reference_prediction_result = reference_prediction_artifact[
    "prediction_result"
]
reference_graph_artifacts = reference_graph_artifact[
    "graph_artifacts"
]

train_cache = load_origin_aligned_token_cache(TRAIN_CACHE_PATH)
validation_cache = load_origin_aligned_token_cache(VALIDATION_CACHE_PATH)
validate_origin_aligned_token_cache(train_cache)
validate_origin_aligned_token_cache(validation_cache)

train_dataset = CachedTokenGraphDataset(
    train_cache,
    source_path=TRAIN_CACHE_PATH,
    validate=False,
)
validation_dataset = CachedTokenGraphDataset(
    validation_cache,
    source_path=VALIDATION_CACHE_PATH,
    validate=False,
)

train_raw, validation_raw, test_raw = load_candle_splits(DATA_DIR)
train_split, validation_split, _ = clean_candle_splits(
    train_raw,
    validation_raw,
    test_raw,
)

seed = int(resolved_config["training"]["seed"])
set_all_seeds(seed)

initial_model_cpu = DynamicGraphTokenForecaster.from_config(
    resolved_config
).cpu().eval()

best_model_cpu = DynamicGraphTokenForecaster.from_config(
    resolved_config
).cpu()
best_model_cpu.load_state_dict(
    best_checkpoint["model_state_dict"],
    strict=True,
)
best_model_cpu.eval()

last_model_cpu = DynamicGraphTokenForecaster.from_config(
    resolved_config
).cpu()
last_model_cpu.load_state_dict(
    last_checkpoint["model_state_dict"],
    strict=True,
)
last_model_cpu.eval()

reference_best_model_cpu = DynamicGraphTokenForecaster.from_config(
    reference_resolved_config
).cpu()
reference_best_model_cpu.load_state_dict(
    reference_best_checkpoint["model_state_dict"],
    strict=True,
)
reference_best_model_cpu.eval()

forecasting_config = load_yaml(FORECASTING_CONFIG_PATH)
asset_cols = list(validation_cache["asset_cols"])
evaluation_horizons = [
    int(value)
    for value in validation_cache["evaluation_horizons"]
]
evaluation_indices = [
    int(value)
    for value in validation_cache["evaluation_indices"]
]

if asset_cols != list(validation_split["asset_cols"]):
    raise AssertionError(
        "Validation cache and raw split asset ordering differ."
    )

if asset_cols != list(prediction_result["asset_cols"]):
    raise AssertionError(
        "Current prediction asset ordering differs from the cache."
    )

if asset_cols != list(reference_prediction_result["asset_cols"]):
    raise AssertionError(
        "Reference prediction asset ordering differs from the cache."
    )

print("Current best epoch:", best_checkpoint["epoch"])
print("Current last epoch:", last_checkpoint["epoch"])
print("Current best primary score:", best_checkpoint["best_score"])
print("Reference best epoch:", reference_best_checkpoint["epoch"])
print("Reference best primary score:", reference_best_checkpoint["best_score"])
print("Training windows:", len(train_dataset))
print("Validation windows:", len(validation_dataset))
print("Assets:", len(asset_cols))
print("Evaluation horizons:", evaluation_horizons)


In [ ]:
# Cell 4 — shared diagnostic helpers.

def collate_cache_items(
    dataset: CachedTokenGraphDataset,
    indices: Sequence[int],
) -> dict[str, Any]:
    items = [dataset[int(index)] for index in indices]
    output: dict[str, Any] = {}

    for key in items[0]:
        values = [item[key] for item in items]
        if torch.is_tensor(values[0]):
            output[key] = torch.stack(values, dim=0)
        else:
            output[key] = values

    return output


def safe_pearson(
    x: Tensor,
    y: Tensor,
    *,
    eps: float = 1.0e-12,
) -> float:
    x = torch.as_tensor(x).detach().cpu().to(torch.float64).reshape(-1)
    y = torch.as_tensor(y).detach().cpu().to(torch.float64).reshape(-1)
    finite = torch.isfinite(x) & torch.isfinite(y)
    x = x[finite]
    y = y[finite]

    if x.numel() < 2:
        return float("nan")

    x = x - x.mean()
    y = y - y.mean()
    denominator = torch.sqrt(x.square().sum() * y.square().sum())

    if denominator <= eps:
        return float("nan")

    return float((x * y).sum() / denominator)


def off_diagonal_mask(num_nodes: int) -> Tensor:
    return ~torch.eye(num_nodes, dtype=torch.bool)


def graph_statistics(name: str, adjacency: Tensor) -> dict[str, Any]:
    graph = torch.as_tensor(adjacency).detach().cpu().to(torch.float64)

    if graph.ndim != 3 or graph.shape[-1] != graph.shape[-2]:
        raise ValueError("adjacency must have shape [G, N, N].")

    num_nodes = int(graph.shape[-1])
    mask = off_diagonal_mask(num_nodes).unsqueeze(0).expand_as(graph)
    off_diagonal = graph[mask]
    positive = graph > 0

    entropy = -torch.where(
        positive,
        graph * graph.clamp_min(1.0e-12).log(),
        torch.zeros_like(graph),
    ).sum(dim=-1)

    sorted_weights = torch.sort(graph, dim=-1, descending=True).values
    source_mass = graph.sum(dim=1)

    return {
        "state": name,
        "mean_row_entropy": float(entropy.mean()),
        "mean_effective_neighbours": float(entropy.exp().mean()),
        "mean_nonzero_sources": float(
            positive.sum(dim=-1).to(torch.float64).mean()
        ),
        "off_diagonal_weight_mean": float(off_diagonal.mean()),
        "off_diagonal_weight_std": float(off_diagonal.std()),
        "minimum_off_diagonal_weight": float(off_diagonal.min()),
        "maximum_edge_weight": float(graph.max()),
        "mean_top_1_row_mass": float(sorted_weights[..., :1].sum(-1).mean()),
        "mean_top_5_row_mass": float(sorted_weights[..., :5].sum(-1).mean()),
        "mean_top_10_row_mass": float(sorted_weights[..., :10].sum(-1).mean()),
        "mean_top_20_row_mass": float(sorted_weights[..., :20].sum(-1).mean()),
        "source_mass_std": float(source_mass.std()),
        "maximum_source_mass": float(source_mass.max()),
        "head_off_diagonal_correlation": (
            safe_pearson(graph[0][mask[0]], graph[1][mask[1]])
            if int(graph.shape[0]) == 2
            else float("nan")
        ),
    }


def graph_pair_statistics(
    first_name: str,
    first_logits: Tensor,
    first_graph: Tensor,
    second_name: str,
    second_logits: Tensor,
    second_graph: Tensor,
) -> dict[str, Any]:
    first_logits = torch.as_tensor(first_logits).detach().cpu().to(torch.float64)
    second_logits = torch.as_tensor(second_logits).detach().cpu().to(torch.float64)
    first_graph = torch.as_tensor(first_graph).detach().cpu().to(torch.float64)
    second_graph = torch.as_tensor(second_graph).detach().cpu().to(torch.float64)

    num_nodes = int(first_graph.shape[-1])
    mask = off_diagonal_mask(num_nodes).unsqueeze(0).expand_as(first_graph)

    logit_difference = second_logits[mask] - first_logits[mask]
    graph_difference = second_graph[mask] - first_graph[mask]

    return {
        "comparison": f"{first_name} -> {second_name}",
        "off_diagonal_logit_l2_change": float(logit_difference.norm()),
        "off_diagonal_logit_rms_change": float(
            logit_difference.norm() / math.sqrt(logit_difference.numel())
        ),
        "off_diagonal_logit_relative_l2_change": float(
            logit_difference.norm()
            / first_logits[mask].norm().clamp_min(1.0e-12)
        ),
        "maximum_absolute_logit_change": float(logit_difference.abs().max()),
        "adjacency_l2_change": float(graph_difference.norm()),
        "adjacency_rms_change": float(
            graph_difference.norm() / math.sqrt(graph_difference.numel())
        ),
        "maximum_absolute_edge_change": float(graph_difference.abs().max()),
        "adjacency_correlation": safe_pearson(
            first_graph[mask],
            second_graph[mask],
        ),
    }


def module_group(parameter_name: str) -> str:
    if parameter_name.startswith("token_embedding."):
        return "token_embedding"
    if parameter_name.startswith("temporal_blocks."):
        return "temporal_encoder"
    if parameter_name.startswith("graph_learners."):
        return "graph_learner"
    if parameter_name.startswith("spatial_blocks."):
        return "spatial_module"
    if "token_heads.s1_classifier" in parameter_name:
        return "s1_classifier"
    if "token_heads.s2_" in parameter_name:
        return "s2_head"
    if parameter_name.startswith("future_predictor."):
        return "future_predictor_shared"
    return "other"


def select_stratified_windows(
    sample_idx: Tensor,
    *,
    max_windows: int,
    seed: int,
) -> list[int]:
    sample_idx = torch.as_tensor(sample_idx).long()
    generator = np.random.default_rng(seed)
    by_session: dict[int, list[int]] = defaultdict(list)

    for window_index, session_index in enumerate(sample_idx.tolist()):
        by_session[int(session_index)].append(int(window_index))

    for values in by_session.values():
        generator.shuffle(values)

    selected: list[int] = []
    sessions = sorted(by_session)

    while len(selected) < min(max_windows, int(sample_idx.numel())):
        added = False
        for session in sessions:
            values = by_session[session]
            if values:
                selected.append(values.pop())
                added = True
                if len(selected) >= max_windows:
                    break
        if not added:
            break

    return sorted(selected)


def token_ids_to_bits(token_ids: Tensor, *, num_bits: int = 10) -> Tensor:
    bit_positions = torch.arange(
        num_bits,
        device=token_ids.device,
        dtype=torch.long,
    )
    return ((token_ids.long().unsqueeze(-1) >> bit_positions) & 1)


def code_usage_summary(name: str, values: Tensor) -> dict[str, Any]:
    counts = torch.bincount(
        values.long().reshape(-1),
        minlength=1024,
    ).to(torch.float64)
    probabilities = counts[counts > 0]
    probabilities = probabilities / probabilities.sum()
    entropy = float(-(probabilities * probabilities.log()).sum())
    sorted_counts = torch.sort(counts, descending=True).values

    return {
        "stream": name,
        "codes_used": int((counts > 0).sum()),
        "effective_vocabulary": math.exp(entropy),
        "largest_code_share": float(sorted_counts[0] / sorted_counts.sum()),
        "top_10_share": float(sorted_counts[:10].sum() / sorted_counts.sum()),
    }


def invalid_candle_mask(values: Tensor) -> Tensor:
    values = torch.as_tensor(values)
    open_values = values[..., 0]
    high_values = values[..., 1]
    low_values = values[..., 2]
    close_values = values[..., 3]
    volume_values = values[..., 4]

    return (
        ~torch.isfinite(values).all(dim=-1)
        | (open_values <= 0)
        | (high_values <= 0)
        | (low_values <= 0)
        | (close_values <= 0)
        | (high_values < torch.maximum(open_values, close_values))
        | (low_values > torch.minimum(open_values, close_values))
        | (high_values < low_values)
        | (volume_values < 0)
    )


def raw_ohlcv_for_window(
    window_index: int,
    *,
    asset_indices: Sequence[int] | None = None,
) -> tuple[Tensor, Tensor]:
    sample_idx = int(validation_cache["sample_idx"][window_index])
    origin_idx = int(validation_cache["origin_idx"][window_index])
    x_day, _, _ = validation_split["samples"][sample_idx]

    channel_indices = [
        get_channel_index(validation_split, channel)
        for channel in ("open", "high", "low", "close", "volume")
    ]

    if asset_indices is None:
        asset_indices = list(range(len(asset_cols)))

    context_length = int(resolved_config["forecasting"]["context_length"])
    context_start = origin_idx - context_length + 1

    context = x_day[context_start : origin_idx + 1][
        :, asset_indices
    ][..., channel_indices]
    future = x_day[origin_idx + 1 : origin_idx + 61][
        :, asset_indices
    ][..., channel_indices]

    return context.to(torch.float32), future.to(torch.float32)


def evaluate_decoded_variant(
    name: str,
    decoded: Tensor,
    actual: Tensor,
    origin_close: Tensor,
) -> dict[str, Any]:
    decoded = decoded.to(torch.float64)
    actual = actual.to(torch.float64)
    predicted_close = decoded[..., 3].clamp_min(1.0e-8)
    actual_close = actual[..., 3].clamp_min(1.0e-8)
    origin_close = origin_close.to(torch.float64).clamp_min(1.0e-8)

    predicted_eval = predicted_close[:, evaluation_indices]
    actual_eval = actual_close[:, evaluation_indices]
    predicted_clg = torch.log(predicted_eval / origin_close[:, None, :])
    actual_clg = torch.log(actual_eval / origin_close[:, None, :])

    predicted_path = torch.cat([origin_close[:, None, :], predicted_close], dim=1)
    actual_path = torch.cat([origin_close[:, None, :], actual_close], dim=1)
    predicted_returns = torch.log(predicted_path[:, 1:] / predicted_path[:, :-1])
    actual_returns = torch.log(actual_path[:, 1:] / actual_path[:, :-1])

    true_magnitude = actual_clg.abs().mean()
    magnitude_ratio = (
        float(predicted_clg.abs().mean() / true_magnitude)
        if true_magnitude > 0
        else float("nan")
    )

    return {
        "variant": name,
        "dense_log_price_mae": float(
            torch.log(predicted_close / actual_close).abs().mean()
        ),
        "five_horizon_clg_mae": float(
            (predicted_clg - actual_clg).abs().mean()
        ),
        "dense_return_pearson": safe_pearson(
            predicted_returns,
            actual_returns,
        ),
        "movement_magnitude_ratio": magnitude_ratio,
        "invalid_candle_rate_percent": float(
            invalid_candle_mask(decoded).to(torch.float64).mean() * 100.0
        ),
    }


def top_edges_table(
    adjacency: Tensor,
    *,
    edges_per_head: int = 20,
) -> pd.DataFrame:
    graph = torch.as_tensor(adjacency).detach().cpu().to(torch.float64)
    rows: list[dict[str, Any]] = []

    for head in range(int(graph.shape[0])):
        matrix = graph[head].clone()
        matrix.fill_diagonal_(0.0)
        flat_indices = torch.topk(
            matrix.reshape(-1),
            k=min(edges_per_head, matrix.numel()),
        ).indices

        for rank, flat_index in enumerate(flat_indices.tolist(), start=1):
            target = flat_index // int(matrix.shape[1])
            source = flat_index % int(matrix.shape[1])
            rows.append(
                {
                    "head": head,
                    "rank": rank,
                    "target": asset_cols[target],
                    "source": asset_cols[source],
                    "weight": float(matrix[target, source]),
                }
            )

    return pd.DataFrame(rows)


In [ ]:
# Cell 5 — artefact alignment and controlled-run integrity audit.

expected_validation_windows = len(validation_dataset)
expected_prediction_shape = (
    expected_validation_windows,
    len(evaluation_horizons),
    len(asset_cols),
    1,
)

for label, result in (
    ("coarse-only", prediction_result),
    ("full-token reference", reference_prediction_result),
):
    if tuple(result["y_pred"].shape) != expected_prediction_shape:
        raise AssertionError(
            f"Unexpected {label} prediction shape."
        )

    if tuple(result["y_true"].shape) != expected_prediction_shape:
        raise AssertionError(
            f"Unexpected {label} target shape."
        )

    if not torch.equal(
        result["sample_idx"],
        validation_cache["sample_idx"],
    ):
        raise AssertionError(
            f"{label} sample_idx differs from the validation cache."
        )

    if not torch.equal(
        result["origin_idx"],
        validation_cache["origin_idx"],
    ):
        raise AssertionError(
            f"{label} origin_idx differs from the validation cache."
        )

    sparse_target_indices = validation_cache[
        "target_indices"
    ][:, evaluation_indices]

    if not torch.equal(
        result["target_indices"],
        sparse_target_indices,
    ):
        raise AssertionError(
            f"{label} target indices are misaligned."
        )

if not torch.equal(
    prediction_result["y_true"],
    reference_prediction_result["y_true"],
):
    raise AssertionError(
        "The controlled runs do not share identical validation targets."
    )

if not torch.equal(
    prediction_result["last_context_target"],
    reference_prediction_result["last_context_target"],
):
    raise AssertionError(
        "The controlled runs do not share identical context anchors."
    )

for key in ("sample_idx", "origin_idx"):
    if not torch.equal(
        prediction_result[key],
        graph_artifacts[key],
    ):
        raise AssertionError(
            f"Current prediction and graph {key} differ."
        )

selected_graphs = torch.as_tensor(
    graph_artifacts["selected"]
)
base_graphs = torch.as_tensor(
    graph_artifacts["base"]
)
logit_artifacts = torch.as_tensor(
    graph_artifacts["logits"]
)

if selected_graphs.shape[0] != expected_validation_windows:
    raise AssertionError(
        "Unexpected graph window dimension."
    )

if not torch.equal(
    selected_graphs,
    selected_graphs[:1].expand_as(selected_graphs),
):
    raise AssertionError(
        "The free-static graph varies across validation windows."
    )

if not torch.equal(
    selected_graphs,
    base_graphs,
):
    raise AssertionError(
        "For free_static, selected and base graphs must match."
    )

best_learner = best_model_cpu.graph_learners[0]
best_checkpoint_logits = best_learner.logits.detach().cpu()
best_checkpoint_graph = (
    best_learner
    .singleton_adjacency()[0]
    .detach()
    .cpu()
)

if not torch.allclose(
    logit_artifacts[0],
    best_checkpoint_logits,
    atol=0.0,
    rtol=0.0,
):
    raise AssertionError(
        "Saved graph logits do not match the best checkpoint."
    )

if not torch.allclose(
    selected_graphs[0],
    best_checkpoint_graph,
    atol=1.0e-7,
    rtol=1.0e-6,
):
    raise AssertionError(
        "Saved selected graph does not match the best checkpoint."
    )

primary_from_table = float(
    metric_table.loc[
        metric_table["metric"]
        == "cumulative_log_change_mae",
        "value",
    ].mean()
)

reference_primary_from_table = float(
    reference_metric_table.loc[
        reference_metric_table["metric"]
        == "cumulative_log_change_mae",
        "value",
    ].mean()
)

if not math.isclose(
    primary_from_table,
    float(saved_diagnostics["primary_score"]),
    rel_tol=0.0,
    abs_tol=1.0e-10,
):
    raise AssertionError(
        "Current primary score does not match the metric table."
    )

if not math.isclose(
    reference_primary_from_table,
    float(reference_diagnostics["primary_score"]),
    rel_tol=0.0,
    abs_tol=1.0e-10,
):
    raise AssertionError(
        "Reference primary score does not match its metric table."
    )

generated_s2_value = float(
    saved_diagnostics["generated_s2_accuracy"]
)

if not math.isnan(generated_s2_value):
    raise AssertionError(
        "Coarse-only generated_s2_accuracy should be NaN."
    )

integrity_summary = pd.DataFrame(
    [
        {
            "item": "current future-token mode",
            "value": saved_diagnostics["future_token_mode"],
        },
        {
            "item": "current best epoch",
            "value": best_checkpoint["epoch"],
        },
        {
            "item": "reference best epoch",
            "value": reference_best_checkpoint["epoch"],
        },
        {
            "item": "validation windows",
            "value": expected_validation_windows,
        },
        {
            "item": "validation sessions",
            "value": int(
                prediction_result["sample_idx"].unique().numel()
            ),
        },
        {
            "item": "assets",
            "value": len(asset_cols),
        },
        {
            "item": "coarse-only primary score",
            "value": saved_diagnostics["primary_score"],
        },
        {
            "item": "full-token reference primary score",
            "value": reference_diagnostics["primary_score"],
        },
        {
            "item": "coarse-only generated s1 accuracy",
            "value": saved_diagnostics["generated_s1_accuracy"],
        },
        {
            "item": "full-token generated s1 accuracy",
            "value": reference_diagnostics["generated_s1_accuracy"],
        },
        {
            "item": "coarse-only generated s2 accuracy",
            "value": "not applicable",
        },
        {
            "item": "coarse-only invalid dense candles (%)",
            "value": saved_diagnostics[
                "invalid_dense_candle_rate_percent"
            ],
        },
        {
            "item": "full-token invalid dense candles (%)",
            "value": reference_diagnostics[
                "invalid_dense_candle_rate_percent"
            ],
        },
    ]
)

display(integrity_summary)
print("COARSE-ONLY ARTEFACT ALIGNMENT AUDIT PASSED")


In [ ]:
# Cell 6 — training history and controlled full-token comparison.

plt.figure(figsize=(9, 5))
plt.plot(
    history["epoch"],
    history["train_token_loss"],
    marker="o",
    label="coarse-only training s1 loss",
)
plt.plot(
    history["epoch"],
    history["validation_token_loss"],
    marker="o",
    label="coarse-only validation s1 loss",
)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Coarse-only token-space optimisation")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(
    history["epoch"],
    history["validation_s1_loss"],
    marker="o",
    label="coarse-only validation s1 CE",
)
plt.plot(
    reference_history["epoch"],
    reference_history["validation_s1_loss"],
    marker="o",
    label="full-token reference validation s1 CE",
)
plt.xlabel("Epoch")
plt.ylabel("s1 cross-entropy")
plt.title("Controlled comparison: validation coarse-token loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(
    history["epoch"],
    history["generated_s1_accuracy"],
    marker="o",
    label="coarse-only generated s1 accuracy",
)
plt.plot(
    reference_history["epoch"],
    reference_history["generated_s1_accuracy"],
    marker="o",
    label="full-token reference generated s1 accuracy",
)
plt.xlabel("Epoch")
plt.ylabel("Exact s1 accuracy")
plt.title("Controlled comparison: generated coarse-token accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(
    history["epoch"],
    history["primary_score"],
    marker="o",
    label="coarse-only primary score",
)
plt.plot(
    reference_history["epoch"],
    reference_history["primary_score"],
    marker="o",
    label="full-token reference primary score",
)
plt.axvline(
    int(best_checkpoint["epoch"]),
    linestyle="--",
    label=f"coarse-only selected epoch {best_checkpoint['epoch']}",
)
plt.xlabel("Epoch")
plt.ylabel("Mean cumulative-log-change MAE")
plt.title("Controlled comparison: decoded validation objective")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(
    history["epoch"],
    history["train_graph_mean_row_entropy"],
    marker="o",
    label="coarse-only training graph entropy",
)
plt.plot(
    history["epoch"],
    history["validation_graph_mean_row_entropy"],
    marker="o",
    label="coarse-only validation graph entropy",
)
plt.plot(
    reference_history["epoch"],
    reference_history["validation_graph_mean_row_entropy"],
    linestyle="--",
    label="full-token reference validation entropy",
)
plt.axhline(
    math.log(len(asset_cols) - 1),
    linestyle=":",
    label="uniform non-self graph entropy",
)
plt.axhline(
    2.2,
    linestyle="-.",
    label="target entropy = 2.2",
)
plt.xlabel("Epoch")
plt.ylabel("Mean row entropy")
plt.title("Graph entropy under the coarse-only objective")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(
    history["epoch"],
    history["train_graph_mean_effective_neighbours"],
    marker="o",
    label="coarse-only training effective neighbours",
)
plt.plot(
    history["epoch"],
    history["validation_graph_mean_effective_neighbours"],
    marker="o",
    label="coarse-only validation effective neighbours",
)
plt.plot(
    reference_history["epoch"],
    reference_history[
        "validation_graph_mean_effective_neighbours"
    ],
    linestyle="--",
    label="full-token reference validation effective neighbours",
)
plt.axhline(
    math.exp(2.2),
    linestyle="-.",
    label=f"target effective neighbours = {math.exp(2.2):.2f}",
)
plt.xlabel("Epoch")
plt.ylabel("Effective neighbours")
plt.title("Graph concentration during training")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(10, 6))
for horizon in evaluation_horizons:
    current_column = f"val/cumulative_log_change_mae/h{horizon}"

    if current_column in history:
        plt.plot(
            history["epoch"],
            history[current_column],
            marker="o",
            label=f"coarse-only h={horizon}",
        )

    if current_column in reference_history:
        plt.plot(
            reference_history["epoch"],
            reference_history[current_column],
            linestyle="--",
            alpha=0.7,
            label=f"full-token h={horizon}",
        )

plt.xlabel("Epoch")
plt.ylabel("Cumulative-log-change MAE")
plt.title("Decoded validation MAE by horizon")
plt.legend(ncol=2)
plt.grid(alpha=0.3)
plt.show()

comparison_metrics = [
    "cumulative_log_change_mae",
    "relative_mae_vs_persistence",
    "persistence_win_rate",
    "cumulative_log_change_cross_sectional_pearson_ic",
    "cumulative_log_change_cross_sectional_spearman_rank_ic",
]

current_comparison = metric_table.loc[
    metric_table["metric"].isin(comparison_metrics),
    ["metric", "horizon", "value"],
].copy()
current_comparison["run"] = "coarse_only"

reference_comparison = reference_metric_table.loc[
    reference_metric_table["metric"].isin(comparison_metrics),
    ["metric", "horizon", "value"],
].copy()
reference_comparison["run"] = "full_token_reference"

controlled_metric_table = pd.concat(
    [current_comparison, reference_comparison],
    ignore_index=True,
).pivot_table(
    index=["metric", "horizon"],
    columns="run",
    values="value",
).sort_index()

controlled_metric_table["coarse_minus_full"] = (
    controlled_metric_table["coarse_only"]
    - controlled_metric_table["full_token_reference"]
)

display(controlled_metric_table)

selected_history_columns = [
    "epoch",
    "train_token_loss",
    "validation_token_loss",
    "primary_score",
    "train_s1_loss",
    "validation_s1_loss",
    "train_s1_accuracy",
    "validation_s1_accuracy",
    "generated_s1_accuracy",
    "train_graph_mean_row_entropy",
    "train_graph_mean_effective_neighbours",
    "train_graph_regularisation_loss",
    "train_graph_target_entropy_penalty",
]

display(
    history.loc[
        history["epoch"].isin(
            sorted(
                {
                    1,
                    int(best_checkpoint["epoch"]),
                    int(last_checkpoint["epoch"]),
                }
            )
        ),
        [
            column
            for column in selected_history_columns
            if column in history
        ],
    ].reset_index(drop=True)
)


In [ ]:
# Cell 7 — initial, current best/last, and full-token-reference graph states.

def extract_free_static_state(
    model: DynamicGraphTokenForecaster,
) -> tuple[Tensor, Tensor]:
    learner = model.graph_learners[0]

    if not hasattr(learner, "logits") or not hasattr(
        learner, "singleton_adjacency"
    ):
        raise TypeError(
            "Model does not contain a free-static graph learner."
        )

    logits = learner.logits.detach().cpu().clone()
    adjacency = (
        learner.singleton_adjacency()[0]
        .detach()
        .cpu()
        .clone()
    )
    return logits, adjacency


initial_logits, initial_graph = extract_free_static_state(
    initial_model_cpu
)
best_logits, best_graph = extract_free_static_state(
    best_model_cpu
)
last_logits, last_graph = extract_free_static_state(
    last_model_cpu
)
reference_best_logits, reference_best_graph = (
    extract_free_static_state(
        reference_best_model_cpu
    )
)

graph_state_table = pd.DataFrame(
    [
        graph_statistics("initial", initial_graph),
        graph_statistics("coarse_only_best", best_graph),
        graph_statistics("coarse_only_last", last_graph),
        graph_statistics(
            "full_token_reference_best",
            reference_best_graph,
        ),
    ]
)

graph_movement_table = pd.DataFrame(
    [
        graph_pair_statistics(
            "initial",
            initial_logits,
            initial_graph,
            "coarse_only_best",
            best_logits,
            best_graph,
        ),
        graph_pair_statistics(
            "initial",
            initial_logits,
            initial_graph,
            "coarse_only_last",
            last_logits,
            last_graph,
        ),
        graph_pair_statistics(
            "coarse_only_best",
            best_logits,
            best_graph,
            "coarse_only_last",
            last_logits,
            last_graph,
        ),
        graph_pair_statistics(
            "full_token_reference_best",
            reference_best_logits,
            reference_best_graph,
            "coarse_only_best",
            best_logits,
            best_graph,
        ),
    ]
)

# Free-static train/eval graph construction should be exactly deterministic.
dummy_hidden = torch.zeros(
    2,
    int(model_config["context_length"]),
    int(model_config["num_nodes"]),
    int(model_config["d_model"]),
)

best_learner = best_model_cpu.graph_learners[0]
best_learner.train()
train_graph_1 = best_learner(dummy_hidden).selected.detach().cpu()
train_graph_2 = best_learner(dummy_hidden).selected.detach().cpu()
best_learner.eval()
evaluation_graph = best_learner(dummy_hidden).selected.detach().cpu()

if not torch.equal(train_graph_1, train_graph_2):
    raise AssertionError("Free-static training graph is stochastic.")

if not torch.equal(train_graph_1, evaluation_graph):
    raise AssertionError("Free-static train and evaluation graphs differ.")

print("Repeated train/eval graph equality: True")
display(graph_state_table)
display(graph_movement_table)

# Compare parameter movement across the complete model. This contextualises
# the free-static graph movement against the temporal, spatial and future
# predictor modules.
def model_parameter_movement(
    first_model: DynamicGraphTokenForecaster,
    second_model: DynamicGraphTokenForecaster,
    *,
    comparison: str,
) -> pd.DataFrame:
    first_parameters = dict(first_model.named_parameters())
    second_parameters = dict(second_model.named_parameters())
    accumulator: dict[str, dict[str, float]] = defaultdict(
        lambda: {
            "difference_squared": 0.0,
            "first_squared": 0.0,
            "num_parameters": 0.0,
            "maximum_absolute_change": 0.0,
        }
    )

    for parameter_name, second_parameter in second_parameters.items():
        first_parameter = first_parameters[parameter_name]
        difference = (
            second_parameter.detach().to(torch.float64)
            - first_parameter.detach().to(torch.float64)
        )
        group = module_group(parameter_name)
        accumulator[group]["difference_squared"] += float(
            difference.square().sum()
        )
        accumulator[group]["first_squared"] += float(
            first_parameter.detach().to(torch.float64).square().sum()
        )
        accumulator[group]["num_parameters"] += int(second_parameter.numel())
        accumulator[group]["maximum_absolute_change"] = max(
            accumulator[group]["maximum_absolute_change"],
            float(difference.abs().max()),
        )

    rows = []
    for group, values in sorted(accumulator.items()):
        difference_norm = math.sqrt(values["difference_squared"])
        first_norm = math.sqrt(values["first_squared"])
        num_parameters = max(int(values["num_parameters"]), 1)
        rows.append(
            {
                "comparison": comparison,
                "module": group,
                "parameters": num_parameters,
                "absolute_l2_change": difference_norm,
                "relative_l2_change": difference_norm / max(first_norm, 1.0e-12),
                "rms_parameter_change": difference_norm / math.sqrt(num_parameters),
                "maximum_absolute_change": values["maximum_absolute_change"],
            }
        )

    return pd.DataFrame(rows)


parameter_movement_table = pd.concat(
    [
        model_parameter_movement(
            initial_model_cpu,
            best_model_cpu,
            comparison="initial -> best",
        ),
        model_parameter_movement(
            initial_model_cpu,
            last_model_cpu,
            comparison="initial -> last",
        ),
    ],
    ignore_index=True,
)

# Read Adam moments from the best checkpoint. Parameter ordering in an
# optimiser state dict follows model.named_parameters() for this runner.
optimizer_state = best_checkpoint["optimizer_state_dict"]
saved_parameter_ids = [
    parameter_id
    for group in optimizer_state["param_groups"]
    for parameter_id in group["params"]
]
named_parameter_list = list(best_model_cpu.named_parameters())

if len(saved_parameter_ids) != len(named_parameter_list):
    raise AssertionError(
        "Optimizer parameter IDs do not align with model parameters."
    )

optimizer_accumulator: dict[str, dict[str, float]] = defaultdict(
    lambda: {
        "first_moment_squared": 0.0,
        "second_moment_sum": 0.0,
        "num_values": 0.0,
    }
)

for parameter_id, (parameter_name, parameter) in zip(
    saved_parameter_ids,
    named_parameter_list,
    strict=True,
):
    state = optimizer_state["state"].get(parameter_id, {})
    first_moment = state.get("exp_avg")
    second_moment = state.get("exp_avg_sq")

    if first_moment is None:
        continue

    group = module_group(parameter_name)
    optimizer_accumulator[group]["first_moment_squared"] += float(
        first_moment.detach().to(torch.float64).square().sum()
    )
    if second_moment is not None:
        optimizer_accumulator[group]["second_moment_sum"] += float(
            second_moment.detach().to(torch.float64).sum()
        )
    optimizer_accumulator[group]["num_values"] += int(parameter.numel())

optimizer_rows = []
for group, values in sorted(optimizer_accumulator.items()):
    num_values = max(int(values["num_values"]), 1)
    optimizer_rows.append(
        {
            "module": group,
            "adam_first_moment_rms": math.sqrt(
                values["first_moment_squared"] / num_values
            ),
            "adam_second_moment_mean": values["second_moment_sum"] / num_values,
        }
    )

optimizer_moment_table = pd.DataFrame(optimizer_rows)

display(
    parameter_movement_table.sort_values(
        ["comparison", "relative_l2_change"],
        ascending=[True, False],
    )
)
display(
    optimizer_moment_table.sort_values(
        "adam_first_moment_rms",
        ascending=False,
    )
)


In [ ]:
# Cell 8 — graph heat maps, differences, source mass, and top edges.

def plot_graph_heatmap(
    adjacency: Tensor,
    *,
    title: str,
    head: int,
    label_stride: int = 6,
) -> None:
    matrix = adjacency[head].detach().cpu().numpy()
    plt.figure(figsize=(10, 9))
    image = plt.imshow(matrix, aspect="auto", interpolation="nearest")
    plt.colorbar(image, label="Edge weight")
    tick_positions = list(range(0, len(asset_cols), label_stride))
    tick_labels = [asset_cols[position] for position in tick_positions]
    plt.xticks(tick_positions, tick_labels, rotation=90)
    plt.yticks(tick_positions, tick_labels)
    plt.xlabel("Source asset")
    plt.ylabel("Target asset")
    plt.title(f"{title} — head {head} (row=target, column=source)")
    plt.tight_layout()
    plt.show()


def plot_graph_difference(
    first: Tensor,
    second: Tensor,
    *,
    title: str,
    head: int,
    label_stride: int = 6,
) -> None:
    matrix = (second[head] - first[head]).detach().cpu().numpy()
    limit = float(np.max(np.abs(matrix)))
    plt.figure(figsize=(10, 9))
    image = plt.imshow(
        matrix,
        aspect="auto",
        interpolation="nearest",
        vmin=-limit,
        vmax=limit,
        cmap="coolwarm",
    )
    plt.colorbar(image, label="Edge-weight change")
    tick_positions = list(range(0, len(asset_cols), label_stride))
    tick_labels = [asset_cols[position] for position in tick_positions]
    plt.xticks(tick_positions, tick_labels, rotation=90)
    plt.yticks(tick_positions, tick_labels)
    plt.xlabel("Source asset")
    plt.ylabel("Target asset")
    plt.title(f"{title} — head {head}")
    plt.tight_layout()
    plt.show()


def plot_source_mass(adjacency: Tensor, *, title: str, head: int) -> None:
    source_mass = adjacency[head].sum(dim=0).detach().cpu().numpy()
    plt.figure(figsize=(14, 5))
    plt.bar(np.arange(len(asset_cols)), source_mass)
    plt.xticks(
        np.arange(len(asset_cols)),
        asset_cols,
        rotation=90,
        fontsize=7,
    )
    plt.xlabel("Source asset")
    plt.ylabel("Total incoming row mass assigned to source")
    plt.title(f"{title} — source mass, head {head}")
    plt.tight_layout()
    plt.show()


for graph_head in range(int(best_graph.shape[0])):
    plot_graph_heatmap(
        best_graph,
        title="Coarse-only best-checkpoint graph",
        head=graph_head,
    )
    plot_graph_difference(
        initial_graph,
        best_graph,
        title="Coarse-only best minus initial graph",
        head=graph_head,
    )
    plot_source_mass(
        best_graph,
        title="Coarse-only best-checkpoint graph",
        head=graph_head,
    )

    plot_graph_difference(
        reference_best_graph,
        best_graph,
        title="Coarse-only best minus full-token reference best",
        head=graph_head,
    )

print("Largest directed edges at the coarse-only best checkpoint")
display(top_edges_table(best_graph, edges_per_head=25))


In [ ]:
# Cell 9 — s1-only forecasting gradients and graph influence on one real batch.

gradient_batch = collate_cache_items(
    train_dataset,
    list(range(GRADIENT_BATCH_SIZE)),
)

context_tokens = gradient_batch["context_tokens"].long()
target_s1 = gradient_batch["target_s1"].long()

gradient_model = copy.deepcopy(best_model_cpu)
gradient_model.train()

gradient_output = gradient_model(
    context_tokens,
    target_s1=target_s1,
)

if gradient_output.s2_logits is not None:
    raise RuntimeError(
        "Coarse-only supervised forward unexpectedly returned s2 logits."
    )

s1_loss = F.cross_entropy(
    gradient_output.s1_logits.reshape(
        -1,
        gradient_output.s1_logits.shape[-1],
    ),
    target_s1.reshape(-1),
)

named_parameters = [
    (name, parameter)
    for name, parameter in gradient_model.named_parameters()
    if parameter.requires_grad
]
parameters = [
    parameter
    for _, parameter in named_parameters
]

s1_gradients = torch.autograd.grad(
    s1_loss,
    parameters,
    retain_graph=False,
    allow_unused=True,
)

gradient_accumulator: dict[str, dict[str, float]] = defaultdict(
    lambda: {
        "s1_squared": 0.0,
        "num_values": 0.0,
        "parameters_with_gradient": 0.0,
    }
)

graph_parameter_diagnostics: dict[str, float] = {}

for (
    (parameter_name, parameter),
    s1_gradient,
) in zip(
    named_parameters,
    s1_gradients,
    strict=True,
):
    group = module_group(parameter_name)
    gradient_accumulator[group]["num_values"] += int(
        parameter.numel()
    )

    if s1_gradient is None:
        continue

    s1_gradient_64 = (
        s1_gradient.detach()
        .to(torch.float64)
    )

    gradient_accumulator[group]["s1_squared"] += float(
        s1_gradient_64.square().sum()
    )
    gradient_accumulator[group][
        "parameters_with_gradient"
    ] += int(parameter.numel())

    if parameter_name == "graph_learners.0.logits":
        mask = (
            off_diagonal_mask(
                int(parameter.shape[-1])
            )
            .unsqueeze(0)
            .expand_as(s1_gradient_64)
        )

        graph_parameter_diagnostics = {
            "s1_off_diagonal_gradient_norm": float(
                s1_gradient_64[mask].norm()
            ),
            "s1_off_diagonal_gradient_rms": float(
                s1_gradient_64[mask].norm()
                / math.sqrt(
                    s1_gradient_64[mask].numel()
                )
            ),
            "maximum_absolute_s1_diagonal_gradient": float(
                torch.diagonal(
                    s1_gradient_64,
                    dim1=-2,
                    dim2=-1,
                ).abs().max()
            ),
        }

if not graph_parameter_diagnostics:
    raise RuntimeError(
        "Free-static graph logits did not receive an s1 gradient."
    )

gradient_rows = []

for group, values in sorted(
    gradient_accumulator.items()
):
    s1_norm = math.sqrt(
        values["s1_squared"]
    )
    num_values = max(
        int(values["num_values"]),
        1,
    )

    gradient_rows.append(
        {
            "module": group,
            "s1_gradient_norm": s1_norm,
            "s1_gradient_rms": (
                s1_norm
                / math.sqrt(num_values)
            ),
            "parameters_with_s1_gradient": int(
                values["parameters_with_gradient"]
            ),
            "total_parameters": num_values,
        }
    )

gradient_table = pd.DataFrame(
    gradient_rows
)
graph_gradient_table = pd.DataFrame(
    [graph_parameter_diagnostics]
)

# Measure how much the spatial module changes the temporal representation.
spatial_model = copy.deepcopy(
    best_model_cpu
).eval()

with torch.inference_mode():
    embedded = spatial_model.token_embedding(
        context_tokens
    )
    pre_spatial = spatial_model.temporal_blocks[0](
        embedded
    )
    graph_output = spatial_model.graph_learners[0](
        pre_spatial
    )
    adjacency = graph_output.selected

    spatial_layer = (
        spatial_model
        .spatial_blocks[0]
        .layers[0]
    )

    (
        batch_size,
        num_steps,
        num_nodes,
        hidden_dim,
    ) = pre_spatial.shape

    values = (
        spatial_layer.value_projection(
            pre_spatial
        )
        .view(
            batch_size,
            num_steps,
            num_nodes,
            spatial_layer.num_heads,
            spatial_layer.head_dim,
        )
        .permute(0, 1, 3, 2, 4)
    )

    messages = aggregate_graph_values(
        adjacency,
        values,
    )
    messages = (
        messages.permute(0, 1, 3, 2, 4)
        .reshape(
            batch_size,
            num_steps,
            num_nodes,
            hidden_dim,
        )
    )
    messages = spatial_layer.output_projection(
        messages
    )
    post_spatial = spatial_model.spatial_blocks[0](
        pre_spatial,
        adjacency,
    )

    # Isolate the effect of learned graph movement by substituting the
    # reconstructed initial graph logits into an otherwise trained model.
    initial_graph_substitution = copy.deepcopy(
        best_model_cpu
    ).eval()

    initial_graph_substitution.graph_learners[
        0
    ].logits.copy_(
        initial_logits
    )

    best_output = best_model_cpu(
        context_tokens,
        target_s1=target_s1,
    )
    initial_graph_output = initial_graph_substitution(
        context_tokens,
        target_s1=target_s1,
    )

spatial_diagnostics = pd.DataFrame(
    [
        {
            "diagnostic": "s1 scalar loss",
            "value": float(s1_loss),
        },
        {
            "diagnostic": (
                "graph-message norm / pre-spatial norm"
            ),
            "value": float(
                messages.norm()
                / pre_spatial.norm().clamp_min(
                    1.0e-12
                )
            ),
        },
        {
            "diagnostic": (
                "post-spatial change / pre-spatial norm"
            ),
            "value": float(
                (
                    post_spatial
                    - pre_spatial
                ).norm()
                / pre_spatial.norm().clamp_min(
                    1.0e-12
                )
            ),
        },
        {
            "diagnostic": "pre/post-spatial cosine",
            "value": float(
                F.cosine_similarity(
                    pre_spatial.reshape(-1),
                    post_spatial.reshape(-1),
                    dim=0,
                )
            ),
        },
        {
            "diagnostic": (
                "mean |s1 logits(best graph)-"
                "s1 logits(initial graph)|"
            ),
            "value": float(
                (
                    best_output.s1_logits
                    - initial_graph_output.s1_logits
                ).abs().mean()
            ),
        },
        {
            "diagnostic": (
                "s1 argmax change rate "
                "from initial->best graph"
            ),
            "value": float(
                (
                    best_output.s1_logits.argmax(
                        dim=-1
                    )
                    != initial_graph_output.s1_logits.argmax(
                        dim=-1
                    )
                )
                .to(torch.float64)
                .mean()
            ),
        },
    ]
)

display(
    gradient_table.sort_values(
        "s1_gradient_rms",
        ascending=False,
    )
)
display(graph_gradient_table)
display(spatial_diagnostics)

# Retain this for the final summary cell.
free_static_graph_gradient_rms = (
    graph_parameter_diagnostics[
        "s1_off_diagonal_gradient_rms"
    ]
)


In [ ]:
# Cell 10 — full-validation s1 diagnostics and probability-gap analysis.

token_diagnostic_indices = select_stratified_windows(
    validation_cache["sample_idx"],
    max_windows=TOKEN_DIAGNOSTIC_WINDOWS,
    seed=DIAGNOSTIC_SEED,
)

if len(token_diagnostic_indices) != len(validation_dataset):
    print(
        "Warning: token diagnostics use a subset of validation windows:",
        len(token_diagnostic_indices),
        "of",
        len(validation_dataset),
    )

diagnostic_model = copy.deepcopy(
    best_model_cpu
).to(MODEL_DEVICE).eval()

num_future_steps = int(
    model_config["heads"]["prediction_length"]
)

metric_names = (
    "count",
    "s1_ce",
    "s1_top1",
    "s1_top5",
    "s1_top10",
    "s1_bit_accuracy",
    "s1_predictive_entropy",
    "true_token_probability",
    "selected_token_probability",
    "probability_gap",
    "true_to_selected_ratio",
    "top1_top2_margin",
    "top5_probability_mass",
    "top10_probability_mass",
    "true_rank",
)

metric_sums = {
    name: torch.zeros(
        num_future_steps,
        dtype=torch.float64,
    )
    for name in metric_names
}

stored_context_tokens = []
stored_true_s1 = []
stored_true_s2 = []
stored_generated_s1 = []

flat_true_probabilities = []
flat_selected_probabilities = []
flat_probability_gaps = []
flat_probability_ratios = []
flat_top1_top2_margins = []
flat_true_ranks = []
flat_argmax_correct = []

with torch.inference_mode():
    for start in range(
        0,
        len(token_diagnostic_indices),
        TOKEN_DIAGNOSTIC_BATCH_SIZE,
    ):
        stop = min(
            start + TOKEN_DIAGNOSTIC_BATCH_SIZE,
            len(token_diagnostic_indices),
        )

        batch_indices = token_diagnostic_indices[
            start:stop
        ]
        batch = collate_cache_items(
            validation_dataset,
            batch_indices,
        )

        context = batch[
            "context_tokens"
        ].to(MODEL_DEVICE)
        true_s1 = batch[
            "target_s1"
        ].to(MODEL_DEVICE)
        true_s2 = batch[
            "target_s2"
        ].to(MODEL_DEVICE)

        teacher_output = diagnostic_model(
            context,
            target_s1=true_s1,
        )
        generated = diagnostic_model.generate(
            context
        )

        if teacher_output.s2_logits is not None:
            raise RuntimeError(
                "Coarse-only supervised forward returned s2 logits."
            )

        if generated.forecast.s2_logits is not None:
            raise RuntimeError(
                "Coarse-only generation returned s2 logits."
            )

        s1_logits = teacher_output.s1_logits.float()
        generated_s1 = generated.token_ids[
            ...,
            0,
        ]

        if not torch.equal(
            generated_s1,
            s1_logits.argmax(dim=-1),
        ):
            raise AssertionError(
                "Structured-parallel coarse-only generation does not "
                "match supervised s1 argmax."
            )

        batch_count = (
            int(context.shape[0])
            * int(context.shape[2])
        )
        metric_sums["count"] += batch_count

        s1_ce = F.cross_entropy(
            s1_logits.reshape(
                -1,
                s1_logits.shape[-1],
            ),
            true_s1.reshape(-1),
            reduction="none",
        ).reshape_as(true_s1)

        metric_sums["s1_ce"] += (
            s1_ce.sum(dim=(0, 2))
            .cpu()
            .to(torch.float64)
        )

        top10_probabilities, top10_ids = torch.topk(
            F.softmax(s1_logits, dim=-1),
            k=10,
            dim=-1,
        )

        for top_k_value in (1, 5, 10):
            correct = (
                top10_ids[
                    ...,
                    :top_k_value,
                ]
                == true_s1.unsqueeze(-1)
            ).any(dim=-1)

            metric_sums[
                f"s1_top{top_k_value}"
            ] += (
                correct.sum(dim=(0, 2))
                .cpu()
                .to(torch.float64)
            )

        s1_bit_accuracy = (
            token_ids_to_bits(generated_s1)
            == token_ids_to_bits(true_s1)
        ).to(torch.float64).mean(dim=-1)

        metric_sums["s1_bit_accuracy"] += (
            s1_bit_accuracy.sum(dim=(0, 2)).cpu()
        )

        s1_log_probabilities = F.log_softmax(
            s1_logits,
            dim=-1,
        )
        s1_probabilities = s1_log_probabilities.exp()

        s1_entropy = -(
            s1_probabilities
            * s1_log_probabilities
        ).sum(dim=-1)

        true_probability = (
            s1_probabilities.gather(
                dim=-1,
                index=true_s1.unsqueeze(-1),
            ).squeeze(-1)
        )

        selected_probability = (
            top10_probabilities[
                ...,
                0,
            ]
        )

        probability_gap = (
            selected_probability
            - true_probability
        )

        probability_ratio = (
            true_probability
            / selected_probability.clamp_min(
                1.0e-12
            )
        )

        top1_top2_margin = (
            top10_probabilities[
                ...,
                0,
            ]
            - top10_probabilities[
                ...,
                1,
            ]
        )

        top5_probability_mass = (
            top10_probabilities[
                ...,
                :5,
            ].sum(dim=-1)
        )

        top10_probability_mass = (
            top10_probabilities.sum(dim=-1)
        )

        true_logit = s1_logits.gather(
            dim=-1,
            index=true_s1.unsqueeze(-1),
        ).squeeze(-1)

        true_rank = (
            (
                s1_logits
                > true_logit.unsqueeze(-1)
            ).sum(dim=-1)
            + 1
        )

        metric_sums["s1_predictive_entropy"] += (
            s1_entropy.sum(dim=(0, 2))
            .cpu()
            .to(torch.float64)
        )
        metric_sums["true_token_probability"] += (
            true_probability.sum(dim=(0, 2))
            .cpu()
            .to(torch.float64)
        )
        metric_sums["selected_token_probability"] += (
            selected_probability.sum(dim=(0, 2))
            .cpu()
            .to(torch.float64)
        )
        metric_sums["probability_gap"] += (
            probability_gap.sum(dim=(0, 2))
            .cpu()
            .to(torch.float64)
        )
        metric_sums["true_to_selected_ratio"] += (
            probability_ratio.sum(dim=(0, 2))
            .cpu()
            .to(torch.float64)
        )
        metric_sums["top1_top2_margin"] += (
            top1_top2_margin.sum(dim=(0, 2))
            .cpu()
            .to(torch.float64)
        )
        metric_sums["top5_probability_mass"] += (
            top5_probability_mass.sum(dim=(0, 2))
            .cpu()
            .to(torch.float64)
        )
        metric_sums["top10_probability_mass"] += (
            top10_probability_mass.sum(dim=(0, 2))
            .cpu()
            .to(torch.float64)
        )
        metric_sums["true_rank"] += (
            true_rank.sum(dim=(0, 2))
            .cpu()
            .to(torch.float64)
        )

        argmax_correct = generated_s1 == true_s1

        flat_true_probabilities.append(
            true_probability.detach().cpu().reshape(-1)
        )
        flat_selected_probabilities.append(
            selected_probability.detach().cpu().reshape(-1)
        )
        flat_probability_gaps.append(
            probability_gap.detach().cpu().reshape(-1)
        )
        flat_probability_ratios.append(
            probability_ratio.detach().cpu().reshape(-1)
        )
        flat_top1_top2_margins.append(
            top1_top2_margin.detach().cpu().reshape(-1)
        )
        flat_true_ranks.append(
            true_rank.detach().cpu().reshape(-1)
        )
        flat_argmax_correct.append(
            argmax_correct.detach().cpu().reshape(-1)
        )

        stored_context_tokens.append(
            context.detach().cpu().to(torch.int16)
        )
        stored_true_s1.append(
            true_s1.detach().cpu().to(torch.int16)
        )
        stored_true_s2.append(
            true_s2.detach().cpu().to(torch.int16)
        )
        stored_generated_s1.append(
            generated_s1.detach().cpu().to(torch.int16)
        )

token_diagnostics = {
    "window_indices": torch.tensor(
        token_diagnostic_indices,
        dtype=torch.long,
    ),
    "context_tokens": torch.cat(
        stored_context_tokens,
        dim=0,
    ),
    "true_s1": torch.cat(
        stored_true_s1,
        dim=0,
    ),
    "true_s2": torch.cat(
        stored_true_s2,
        dim=0,
    ),
    "generated_s1": torch.cat(
        stored_generated_s1,
        dim=0,
    ),
}

counts = metric_sums[
    "count"
].clamp_min(1.0)

train_target_s1 = train_cache[
    "target_s1"
].long()

true_s1_subset = token_diagnostics[
    "true_s1"
].long()

per_step_rows = []

for step in range(num_future_steps):
    training_s1_counts = torch.bincount(
        train_target_s1[
            :,
            step,
        ].reshape(-1),
        minlength=1024,
    ).to(torch.float64)

    marginal_s1_mode = int(
        training_s1_counts.argmax()
    )

    per_step_rows.append(
        {
            "future_step": step + 1,
            "is_evaluation_horizon": (
                step + 1
                in evaluation_horizons
            ),
            "s1_ce": float(
                metric_sums["s1_ce"][step]
                / counts[step]
            ),
            "s1_top1": float(
                metric_sums["s1_top1"][step]
                / counts[step]
            ),
            "s1_top5": float(
                metric_sums["s1_top5"][step]
                / counts[step]
            ),
            "s1_top10": float(
                metric_sums["s1_top10"][step]
                / counts[step]
            ),
            "s1_bit_accuracy": float(
                metric_sums["s1_bit_accuracy"][step]
                / counts[step]
            ),
            "s1_predictive_entropy": float(
                metric_sums[
                    "s1_predictive_entropy"
                ][step]
                / counts[step]
            ),
            "mean_true_token_probability": float(
                metric_sums[
                    "true_token_probability"
                ][step]
                / counts[step]
            ),
            "mean_selected_token_probability": float(
                metric_sums[
                    "selected_token_probability"
                ][step]
                / counts[step]
            ),
            "mean_probability_gap": float(
                metric_sums[
                    "probability_gap"
                ][step]
                / counts[step]
            ),
            "mean_true_to_selected_ratio": float(
                metric_sums[
                    "true_to_selected_ratio"
                ][step]
                / counts[step]
            ),
            "mean_top1_top2_margin": float(
                metric_sums[
                    "top1_top2_margin"
                ][step]
                / counts[step]
            ),
            "mean_top5_probability_mass": float(
                metric_sums[
                    "top5_probability_mass"
                ][step]
                / counts[step]
            ),
            "mean_top10_probability_mass": float(
                metric_sums[
                    "top10_probability_mass"
                ][step]
                / counts[step]
            ),
            "mean_true_rank": float(
                metric_sums["true_rank"][step]
                / counts[step]
            ),
            "training_marginal_s1_mode_accuracy": float(
                (
                    true_s1_subset[
                        :,
                        step,
                    ]
                    == marginal_s1_mode
                )
                .to(torch.float64)
                .mean()
            ),
        }
    )

token_step_table = pd.DataFrame(
    per_step_rows
)

evaluation_token_table = (
    token_step_table.loc[
        token_step_table[
            "is_evaluation_horizon"
        ]
    ]
    .reset_index(drop=True)
)

display(evaluation_token_table)

# The full-validation recomputation should match the saved best-epoch table.
if len(token_diagnostic_indices) == len(validation_dataset):
    saved_s1 = (
        saved_token_metric_table
        .set_index("future_step")
    )

    maximum_saved_accuracy_difference = float(
        np.max(
            np.abs(
                token_step_table[
                    "s1_top1"
                ].to_numpy()
                - saved_s1.loc[
                    token_step_table[
                        "future_step"
                    ],
                    "generated_s1_accuracy",
                ].to_numpy()
            )
        )
    )

    if maximum_saved_accuracy_difference > 1.0e-7:
        raise AssertionError(
            "Recomputed full-validation s1 accuracy differs "
            "from the saved best-epoch table. Maximum difference: "
            f"{maximum_saved_accuracy_difference}."
        )

plt.figure(figsize=(10, 5))
plt.plot(
    token_step_table["future_step"],
    token_step_table["s1_top1"],
    label="s1 top-1",
)
plt.plot(
    token_step_table["future_step"],
    token_step_table["s1_top5"],
    label="s1 top-5",
)
plt.plot(
    token_step_table["future_step"],
    token_step_table["s1_top10"],
    label="s1 top-10",
)
plt.plot(
    token_step_table["future_step"],
    token_step_table[
        "training_marginal_s1_mode_accuracy"
    ],
    label="training marginal-mode baseline",
)
for horizon in evaluation_horizons:
    plt.axvline(
        horizon,
        linestyle="--",
        alpha=0.25,
    )
plt.xlabel("Future minute")
plt.ylabel("Accuracy")
plt.title("Coarse-token accuracy and top-k coverage")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(
    token_step_table["future_step"],
    token_step_table[
        "mean_selected_token_probability"
    ],
    label="selected argmax-token probability",
)
plt.plot(
    token_step_table["future_step"],
    token_step_table[
        "mean_true_token_probability"
    ],
    label="true-token probability",
)
plt.plot(
    token_step_table["future_step"],
    token_step_table[
        "mean_top1_top2_margin"
    ],
    label="top-1 minus top-2 probability",
)
for horizon in evaluation_horizons:
    plt.axvline(
        horizon,
        linestyle="--",
        alpha=0.25,
    )
plt.xlabel("Future minute")
plt.ylabel("Probability")
plt.title(
    "Probability assigned to the true and selected coarse tokens"
)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

code_usage_table = pd.DataFrame(
    [
        code_usage_summary(
            "true s1",
            token_diagnostics["true_s1"],
        ),
        code_usage_summary(
            "generated s1",
            token_diagnostics["generated_s1"],
        ),
    ]
)

display(code_usage_table)

flat_true_probability = torch.cat(
    flat_true_probabilities
)
flat_selected_probability = torch.cat(
    flat_selected_probabilities
)
flat_probability_gap = torch.cat(
    flat_probability_gaps
)
flat_probability_ratio = torch.cat(
    flat_probability_ratios
)
flat_top1_top2_margin = torch.cat(
    flat_top1_top2_margins
)
flat_true_rank = torch.cat(
    flat_true_ranks
)
flat_correct = torch.cat(
    flat_argmax_correct
)

condition_masks = {
    "all positions": torch.ones_like(
        flat_correct,
        dtype=torch.bool,
    ),
    "argmax correct": flat_correct,
    "argmax wrong": ~flat_correct,
    "true rank 2-5": (
        (flat_true_rank >= 2)
        & (flat_true_rank <= 5)
    ),
    "true rank 6-10": (
        (flat_true_rank >= 6)
        & (flat_true_rank <= 10)
    ),
    "true rank >10": flat_true_rank > 10,
}

condition_rows = []

for condition_name, mask in condition_masks.items():
    positions = int(mask.sum())

    if positions == 0:
        continue

    condition_rows.append(
        {
            "condition": condition_name,
            "positions": positions,
            "share_of_positions": float(
                mask.to(torch.float64).mean()
            ),
            "mean_true_token_probability": float(
                flat_true_probability[mask].mean()
            ),
            "mean_selected_token_probability": float(
                flat_selected_probability[mask].mean()
            ),
            "mean_probability_gap": float(
                flat_probability_gap[mask].mean()
            ),
            "mean_true_to_selected_ratio": float(
                flat_probability_ratio[mask].mean()
            ),
            "mean_top1_top2_margin": float(
                flat_top1_top2_margin[mask].mean()
            ),
            "mean_true_rank": float(
                flat_true_rank[mask]
                .to(torch.float64)
                .mean()
            ),
        }
    )

s1_probability_condition_table = pd.DataFrame(
    condition_rows
)

display(s1_probability_condition_table)

# Direct saved-table comparison at the five dissertation horizons.
current_saved_s1 = saved_token_metric_table.loc[
    saved_token_metric_table[
        "is_evaluation_horizon"
    ],
    [
        "future_step",
        "teacher_forced_s1_ce",
        "generated_s1_accuracy",
    ],
].copy()
current_saved_s1["run"] = "coarse_only"

reference_saved_s1 = reference_token_metric_table.loc[
    reference_token_metric_table[
        "is_evaluation_horizon"
    ],
    [
        "future_step",
        "teacher_forced_s1_ce",
        "generated_s1_accuracy",
    ],
].copy()
reference_saved_s1["run"] = "full_token_reference"

saved_s1_comparison = pd.concat(
    [
        current_saved_s1,
        reference_saved_s1,
    ],
    ignore_index=True,
).pivot(
    index="future_step",
    columns="run",
    values=[
        "teacher_forced_s1_ce",
        "generated_s1_accuracy",
    ],
)

display(saved_s1_comparison)


In [ ]:
# Cell 11 — frozen-decoder analysis for the coarse-only model.

tokenizer = KronosTokenizerAdapter.from_config(
    forecasting_config,
    series_batch_size=TOKENIZER_SERIES_BATCH_SIZE,
).load()

rng = np.random.default_rng(
    DIAGNOSTIC_SEED
)

decode_window_positions = np.sort(
    rng.choice(
        len(token_diagnostic_indices),
        size=min(
            DECODE_DIAGNOSTIC_WINDOWS,
            len(token_diagnostic_indices),
        ),
        replace=False,
    )
)

decode_asset_indices = np.sort(
    rng.choice(
        len(asset_cols),
        size=min(
            DECODE_DIAGNOSTIC_ASSETS,
            len(asset_cols),
        ),
        replace=False,
    )
)

decode_window_positions_list = (
    decode_window_positions.tolist()
)
decode_asset_indices_list = (
    decode_asset_indices.tolist()
)

decode_window_indices = (
    token_diagnostics[
        "window_indices"
    ][decode_window_positions_list]
    .tolist()
)

context_tokens_small = token_diagnostics[
    "context_tokens"
][decode_window_positions_list][
    :,
    :,
    decode_asset_indices_list,
].long()

true_s1_small = token_diagnostics[
    "true_s1"
][decode_window_positions_list][
    :,
    :,
    decode_asset_indices_list,
].long()

true_s2_small = token_diagnostics[
    "true_s2"
][decode_window_positions_list][
    :,
    :,
    decode_asset_indices_list,
].long()

generated_s1_small = token_diagnostics[
    "generated_s1"
][decode_window_positions_list][
    :,
    :,
    decode_asset_indices_list,
].long()

mean_small = validation_cache[
    "context_mean"
][decode_window_indices][
    :,
    decode_asset_indices_list,
]

std_small = validation_cache[
    "context_std"
][decode_window_indices][
    :,
    decode_asset_indices_list,
]

actual_future_parts = []
origin_close_parts = []

for window_index in decode_window_indices:
    raw_context, raw_future = raw_ohlcv_for_window(
        int(window_index),
        asset_indices=decode_asset_indices_list,
    )
    actual_future_parts.append(raw_future)
    origin_close_parts.append(
        raw_context[-1, :, 3]
    )

actual_future_small = torch.stack(
    actual_future_parts,
    dim=0,
)
origin_close_small = torch.stack(
    origin_close_parts,
    dim=0,
)

true_future_pairs = torch.stack(
    [
        true_s1_small,
        true_s2_small,
    ],
    dim=-1,
)

generated_s1_true_s2_pairs = torch.stack(
    [
        generated_s1_small,
        true_s2_small,
    ],
    dim=-1,
)

decoded_variants: dict[str, Tensor] = {}

decoded_variants[
    "oracle full tokens"
] = tokenizer.decode_token_path(
    context_tokens_small,
    true_future_pairs,
    mean=mean_small,
    std=std_small,
    series_batch_size=TOKENIZER_SERIES_BATCH_SIZE,
)

decoded_variants[
    "oracle coarse only"
] = tokenizer.decode_coarse_token_path(
    context_tokens_small,
    true_s1_small,
    mean=mean_small,
    std=std_small,
    series_batch_size=TOKENIZER_SERIES_BATCH_SIZE,
)

decoded_variants[
    "coarse-only model prediction"
] = tokenizer.decode_coarse_token_path(
    context_tokens_small,
    generated_s1_small,
    mean=mean_small,
    std=std_small,
    series_batch_size=TOKENIZER_SERIES_BATCH_SIZE,
)

# Counterfactual upper bound: retain the model's predicted coarse token,
# but supply the true fine token. This is not an available forecast.
decoded_variants[
    "model s1 + true s2 counterfactual"
] = tokenizer.decode_token_path(
    context_tokens_small,
    generated_s1_true_s2_pairs,
    mean=mean_small,
    std=std_small,
    series_batch_size=TOKENIZER_SERIES_BATCH_SIZE,
)

decoded_variant_table = pd.DataFrame(
    [
        evaluate_decoded_variant(
            name,
            decoded,
            actual_future_small,
            origin_close_small,
        )
        for name, decoded in decoded_variants.items()
    ]
).sort_values(
    "five_horizon_clg_mae"
)

display(decoded_variant_table)

print(
    "Diagnostic windows:",
    decode_window_indices,
)
print(
    "Diagnostic assets:",
    [
        asset_cols[index]
        for index in decode_asset_indices_list
    ],
)


In [ ]:
# Cell 12 — draw a new random asset/window price path on every call.

def plot_random_forecast_path() -> dict[str, Any]:
    random_generator = np.random.default_rng()

    window_index = int(
        random_generator.integers(
            0,
            len(validation_dataset),
        )
    )
    asset_index = int(
        random_generator.integers(
            0,
            len(asset_cols),
        )
    )

    batch = collate_cache_items(
        validation_dataset,
        [window_index],
    )

    with torch.inference_mode():
        generated = diagnostic_model.generate(
            batch[
                "context_tokens"
            ].to(MODEL_DEVICE)
        )

    generated_s1_one = (
        generated.token_ids[
            :,
            :,
            asset_index:
            asset_index + 1,
            0,
        ]
        .detach()
        .cpu()
        .long()
    )

    context_tokens_one = batch[
        "context_tokens"
    ][
        :,
        :,
        asset_index:
        asset_index + 1,
    ].long()

    true_s1_one = batch[
        "target_s1"
    ][
        :,
        :,
        asset_index:
        asset_index + 1,
    ].long()

    true_s2_one = batch[
        "target_s2"
    ][
        :,
        :,
        asset_index:
        asset_index + 1,
    ].long()

    true_token_pairs_one = torch.stack(
        [
            true_s1_one,
            true_s2_one,
        ],
        dim=-1,
    )

    mean_one = batch[
        "context_mean"
    ][
        :,
        asset_index:
        asset_index + 1,
    ]
    std_one = batch[
        "context_std"
    ][
        :,
        asset_index:
        asset_index + 1,
    ]

    predicted_coarse = tokenizer.decode_coarse_token_path(
        context_tokens_one,
        generated_s1_one,
        mean=mean_one,
        std=std_one,
        series_batch_size=1,
    )

    oracle_coarse = tokenizer.decode_coarse_token_path(
        context_tokens_one,
        true_s1_one,
        mean=mean_one,
        std=std_one,
        series_batch_size=1,
    )

    oracle_full = tokenizer.decode_token_path(
        context_tokens_one,
        true_token_pairs_one,
        mean=mean_one,
        std=std_one,
        series_batch_size=1,
    )

    raw_context, raw_future = raw_ohlcv_for_window(
        window_index,
        asset_indices=[asset_index],
    )

    actual_close = raw_future[
        :,
        0,
        3,
    ].numpy()

    origin_close = float(
        raw_context[
            -1,
            0,
            3,
        ]
    )

    predicted_coarse_close = (
        predicted_coarse[
            0,
            :,
            0,
            3,
        ]
        .cpu()
        .numpy()
    )

    oracle_coarse_close = (
        oracle_coarse[
            0,
            :,
            0,
            3,
        ]
        .cpu()
        .numpy()
    )

    oracle_full_close = (
        oracle_full[
            0,
            :,
            0,
            3,
        ]
        .cpu()
        .numpy()
    )

    saved_eval_close = (
        prediction_result[
            "y_pred"
        ][
            window_index,
            :,
            asset_index,
            0,
        ]
        .cpu()
        .numpy()
    )

    reference_saved_eval_close = (
        reference_prediction_result[
            "y_pred"
        ][
            window_index,
            :,
            asset_index,
            0,
        ]
        .cpu()
        .numpy()
    )

    future_minutes = np.arange(
        1,
        61,
    )

    plt.figure(
        figsize=(12, 6)
    )
    plt.plot(
        future_minutes,
        actual_close,
        label="actual raw Close",
    )
    plt.plot(
        future_minutes,
        predicted_coarse_close,
        label="coarse-only model path",
    )
    plt.plot(
        future_minutes,
        oracle_coarse_close,
        label="true-s1 coarse-decoder ceiling",
    )
    plt.plot(
        future_minutes,
        oracle_full_close,
        label="true-token full-decoder ceiling",
    )
    plt.plot(
        future_minutes,
        np.full(
            60,
            origin_close,
        ),
        linestyle="--",
        label="persistence",
    )
    plt.scatter(
        evaluation_horizons,
        saved_eval_close,
        marker="x",
        s=70,
        label="saved coarse-only forecasts",
    )
    plt.scatter(
        evaluation_horizons,
        reference_saved_eval_close,
        marker="+",
        s=70,
        label="saved full-token reference forecasts",
    )

    for horizon in evaluation_horizons:
        plt.axvline(
            horizon,
            linestyle=":",
            alpha=0.2,
        )

    sample_idx = int(
        validation_cache[
            "sample_idx"
        ][window_index]
    )
    origin_idx = int(
        validation_cache[
            "origin_idx"
        ][window_index]
    )
    date = validation_cache[
        "dates"
    ][window_index]

    plt.xlabel(
        "Minutes after forecast origin"
    )
    plt.ylabel(
        "Close price"
    )
    plt.title(
        f"{asset_cols[asset_index]} — {date} — "
        f"window {window_index}, session {sample_idx}, "
        f"origin index {origin_idx}"
    )
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    regenerated_eval = (
        predicted_coarse_close[
            evaluation_indices
        ]
    )

    return {
        "window_index": window_index,
        "asset_index": asset_index,
        "asset": asset_cols[asset_index],
        "date": date,
        "sample_idx": sample_idx,
        "origin_idx": origin_idx,
        "origin_close": origin_close,
        "saved_vs_regenerated_eval_max_abs_difference": float(
            np.max(
                np.abs(
                    saved_eval_close
                    - regenerated_eval
                )
            )
        ),
        "coarse_vs_full_reference_saved_eval_mean_abs_difference": float(
            np.mean(
                np.abs(
                    saved_eval_close
                    - reference_saved_eval_close
                )
            )
        ),
    }


random_forecast_details = (
    plot_random_forecast_path()
)

display(
    pd.DataFrame(
        [random_forecast_details]
    )
)

# Rerun this cell, or call plot_random_forecast_path(), for a new draw.


In [ ]:
# Cell 13 — compact controlled-ablation evidence table.

best_graph_stats = graph_state_table.loc[
    graph_state_table["state"]
    == "coarse_only_best"
].iloc[0]

last_graph_stats = graph_state_table.loc[
    graph_state_table["state"]
    == "coarse_only_last"
].iloc[0]

reference_graph_stats = graph_state_table.loc[
    graph_state_table["state"]
    == "full_token_reference_best"
].iloc[0]

initial_to_best = graph_movement_table.loc[
    graph_movement_table["comparison"]
    == "initial -> coarse_only_best"
].iloc[0]

coarse_decoded_row = decoded_variant_table.loc[
    decoded_variant_table["variant"]
    == "coarse-only model prediction"
].iloc[0]

counterfactual_row = decoded_variant_table.loc[
    decoded_variant_table["variant"]
    == "model s1 + true s2 counterfactual"
].iloc[0]

generated_usage_row = code_usage_table.loc[
    code_usage_table["stream"]
    == "generated s1"
].iloc[0]

all_probability_row = (
    s1_probability_condition_table.loc[
        s1_probability_condition_table[
            "condition"
        ]
        == "all positions"
    ].iloc[0]
)

current_primary = float(
    saved_diagnostics["primary_score"]
)
reference_primary = float(
    reference_diagnostics["primary_score"]
)

current_generated_s1 = float(
    saved_diagnostics["generated_s1_accuracy"]
)
reference_generated_s1 = float(
    reference_diagnostics["generated_s1_accuracy"]
)

current_s1_horizon_ce = float(
    current_saved_s1[
        "teacher_forced_s1_ce"
    ].mean()
)
reference_s1_horizon_ce = float(
    reference_saved_s1[
        "teacher_forced_s1_ce"
    ].mean()
)

summary_table = pd.DataFrame(
    [
        {
            "question": (
                "Does removing s2 improve the decoded "
                "five-horizon validation score?"
            ),
            "diagnostic": (
                current_primary
                / reference_primary
            ),
            "interpretation": (
                "<1 favours coarse-only; "
                "1 means no change."
            ),
        },
        {
            "question": (
                "Does removing s2 improve exact s1 accuracy?"
            ),
            "diagnostic": (
                current_generated_s1
                - reference_generated_s1
            ),
            "interpretation": (
                "Absolute full-validation accuracy change."
            ),
        },
        {
            "question": (
                "Does removing s2 improve s1 CE "
                "at [1,5,15,30,60]?"
            ),
            "diagnostic": (
                current_s1_horizon_ce
                / reference_s1_horizon_ce
            ),
            "interpretation": (
                "<1 favours coarse-only."
            ),
        },
        {
            "question": (
                "How many s1 codes does hard argmax use?"
            ),
            "diagnostic": int(
                generated_usage_row["codes_used"]
            ),
            "interpretation": (
                "Measured across the configured "
                "validation diagnostic set."
            ),
        },
        {
            "question": (
                "How large is the average selected-vs-true "
                "probability gap?"
            ),
            "diagnostic": float(
                all_probability_row[
                    "mean_probability_gap"
                ]
            ),
            "interpretation": (
                "Smaller gaps make sampling more plausible."
            ),
        },
        {
            "question": (
                "What fraction of selected-token probability "
                "is assigned to the true token?"
            ),
            "diagnostic": float(
                all_probability_row[
                    "mean_true_to_selected_ratio"
                ]
            ),
            "interpretation": (
                "1 means equal probability on average."
            ),
        },
        {
            "question": (
                "Does s1 forecasting loss reach graph logits?"
            ),
            "diagnostic": (
                free_static_graph_gradient_rms
            ),
            "interpretation": (
                "Off-diagonal s1-loss graph-gradient RMS."
            ),
        },
        {
            "question": (
                "Did the graph move from initialisation "
                "by the selected epoch?"
            ),
            "diagnostic": float(
                initial_to_best[
                    "off_diagonal_logit_relative_l2_change"
                ]
            ),
            "interpretation": (
                "Relative off-diagonal graph-logit L2 movement."
            ),
        },
        {
            "question": (
                "How concentrated is the coarse-only "
                "selected graph?"
            ),
            "diagnostic": float(
                best_graph_stats[
                    "mean_effective_neighbours"
                ]
            ),
            "interpretation": (
                "Uniform non-self graph = 92; target ≈ 9."
            ),
        },
        {
            "question": (
                "How does graph concentration compare with "
                "the full-token reference?"
            ),
            "diagnostic": float(
                best_graph_stats[
                    "mean_effective_neighbours"
                ]
                - reference_graph_stats[
                    "mean_effective_neighbours"
                ]
            ),
            "interpretation": (
                "Negative means the coarse-only graph "
                "is more concentrated."
            ),
        },
        {
            "question": (
                "Would true fine tokens materially improve "
                "the model's predicted s1 path?"
            ),
            "diagnostic": float(
                counterfactual_row[
                    "five_horizon_clg_mae"
                ]
                / coarse_decoded_row[
                    "five_horizon_clg_mae"
                ]
            ),
            "interpretation": (
                "<1 means perfect fine tokens would help."
            ),
        },
    ]
)

display(summary_table)

controlled_run_summary = pd.DataFrame(
    [
        {
            "run": "full_token_reference",
            "best_epoch": int(
                reference_best_checkpoint["epoch"]
            ),
            "primary_score": reference_primary,
            "generated_s1_accuracy": (
                reference_generated_s1
            ),
            "invalid_dense_candle_rate_percent": (
                reference_diagnostics[
                    "invalid_dense_candle_rate_percent"
                ]
            ),
            "graph_effective_neighbours": float(
                reference_graph_stats[
                    "mean_effective_neighbours"
                ]
            ),
        },
        {
            "run": "coarse_only",
            "best_epoch": int(
                best_checkpoint["epoch"]
            ),
            "primary_score": current_primary,
            "generated_s1_accuracy": (
                current_generated_s1
            ),
            "invalid_dense_candle_rate_percent": (
                saved_diagnostics[
                    "invalid_dense_candle_rate_percent"
                ]
            ),
            "graph_effective_neighbours": float(
                best_graph_stats[
                    "mean_effective_neighbours"
                ]
            ),
        },
    ]
)

display(controlled_run_summary)

print(
    "COARSE-ONLY CONTROLLED ABLATION DIAGNOSTIC NOTEBOOK COMPLETE"
)
